In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CACHE_PATH = PROJECT_ROOT / "cache"
CACHE_PATH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

# Note: stockstats is no longer imported here. Technical indicators are now
# calculated in Holidays_and_Indices.ipynb and included in indices_from_2000.csv.
# Rolling returns/volatility are also lagged in the acquisition notebooks.

from index_ticker import SP500, US_INDEX_TICKERS, MACRO_TICKERS, INT_TICKERS

# Note: STOCKSTATS_TECHNICALS is no longer needed in this notebook.
# Technical indicators are pre-calculated and lagged in Holidays_and_Indices.ipynb.

START_DATE = "2000-01-01"
END_DATE = "2026-06-30"

In [2]:
# Read indices_from_2000.csv from cache
df_stock = pd.read_csv(CACHE_PATH / "indices_from_2000.csv", header=[0,1], index_col=0)

In [3]:
# FOMC calendar is no longer loaded here.
# days_since_fomc is now calculated in Holidays_and_Indices.ipynb and included
# in indices_from_2000.csv as ("days_since_fomc", "").
# 
# Kept for reference (commented out):
# fomc = pd.read_csv(CACHE_PATH / "fomc_calendar_2000_present.csv")

In [4]:
# CPI derivative -> Inflation rate (2% - Golden number)

macro = pd.read_csv(CACHE_PATH / "macro_data.csv")

In [5]:
# ============================================================================
# REMOVED: days_since_fomc Calculation
# ============================================================================
# This logic has been moved to Holidays_and_Indices.ipynb and is now part of
# the indices_from_2000.csv cache file. The column ("days_since_fomc", "")
# is automatically included when loading indices_from_2000.csv.
#
# Original code (kept for reference, no longer needed):
# ============================================================================
# Ensure datetime
df_stock = df_stock.copy()
df_stock.index = pd.to_datetime(df_stock.index)

# NOTE: days_since_fomc is now pre-calculated in Holidays_and_Indices.ipynb
# and included in indices_from_2000.csv. No need to recalculate it here.

In [6]:
# Join macro data (with lagged daily macro variables already included)
# Note: macro_data now includes lagged versions of daily FRED series
# (DFF_lag1, DGS2_lag1, DGS10_lag1, DCOILWTICO_lag1 and their pct_change variants)
macro.columns = pd.MultiIndex.from_product( # To match multi-index used by df_stock
    [["Macro"], macro.columns],
    names=df_stock.columns.names,
)

df = df_stock.join(macro, how="left")

In [ ]:
# ============================================================================
# REMOVED: StockStats Technical Indicator Calculation
# ============================================================================
# This logic has been moved to Holidays_and_Indices.ipynb. All 54 technical
# indicators are now calculated there and included in indices_from_2000.csv as
# ("Technical", indicator_name) with lagged versions (e.g., ("Technical", "macd_lag1")).
#
# The indicators are also automatically lagged by 1 trading day to prevent look-ahead
# bias, since they include data through trading day T but predictions are made before
# the market opens on day T.
# 
# NOTE: df_stock already contains technical indicators from indices_from_2000.csv.
# When df_stock is joined with macro data above, all technical indicators are present.

1. Add a few derived macro features.
2. Construct the target.
3. Remove columns that should not be predictors.
4. Remove rows with unavailable features.
5. Fit a regularized baseline using an expanding time-series split.

In [8]:
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ============================================================================
# DERIVE ADDITIONAL FEATURES & PREPARE FOR MODELING
# ============================================================================
# Note: Most feature engineering is now done in the data acquisition notebooks
# (Holidays_and_Indices.ipynb and Macro_FRED.ipynb). The derived features below
# are task-specific calculations for this particular modeling setup.
# ============================================================================

# Yield spread (10Y - 2Y Treasury) using lagged macro variables
# Use DGS10_lag1 and DGS2_lag1 to avoid look-ahead bias
if ("Macro", "DGS10_lag1") in df.columns and ("Macro", "DGS2_lag1") in df.columns:
    df[("Macro", "YieldSpread")] = (
        df[("Macro", "DGS10_lag1")]
        - df[("Macro", "DGS2_lag1")]
    )
elif ("Macro", "DGS10") in df.columns and ("Macro", "DGS2") in df.columns:
    # Fallback to non-lagged if lagged versions not available
    df[("Macro", "YieldSpread")] = (
        df[("Macro", "DGS10")]
        - df[("Macro", "DGS2")]
    )

# Real Fed Funds Rate (DFF - Core Inflation) using lagged daily macro
# Note: Core inflation is lower-frequency so not lagged
if (
    ("Macro", "DFF_lag1") in df.columns
    and ("Macro", "CORESTICKM159SFRBATL") in df.columns
):
    df[("Macro", "RealFedFunds")] = (
        df[("Macro", "DFF_lag1")]
        - df[("Macro", "CORESTICKM159SFRBATL")]
    )
elif (
    ("Macro", "DFF") in df.columns
    and ("Macro", "CORESTICKM159SFRBATL") in df.columns
):
    # Fallback to non-lagged if lagged version not available
    df[("Macro", "RealFedFunds")] = (
        df[("Macro", "DFF")]
        - df[("Macro", "CORESTICKM159SFRBATL")]
    )

# VVIX / VIX ratio (market volatility of volatility) using lagged market indices
# Note: VVIX only available from 2006+, so this may be sparse
if (
    ("Close", "^VVIX") in df.columns
    and ("Close", "^VIX") in df.columns
):
    df[("Macro", "VVIX_VIX")] = (
        df[("Close", "^VVIX")]
        / df[("Close", "^VIX")]
    )


# ============================================================================
# PREPARE FEATURES FOR MODELING
# ============================================================================
# Target: Next trading day's S&P 500 return
TARGET = ("Ret_1", SP500)

START_DATE = "2001-01-01"
y = df.loc[df.index >= pd.Timestamp(START_DATE), TARGET].shift(-1)

# Remove OHLCV columns (we use rolling returns, not raw prices)
remove_first_level = {
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
}

X = df.loc[
    :,
    ~df.columns.get_level_values(0).isin(remove_first_level)
]

# Flatten MultiIndex column names for sklearn compatibility
X.columns = [
    f"{level0}_{level1}".rstrip("_")
    for level0, level1 in X.columns
]

X = X.dropna(axis=1, how="all")

Drop VVIX-derived features to train on the full sample.

In [9]:
# Drop VVIX-derived features to train on the full sample
# VVIX is only available from 2006 onwards, creating sparsity in early data
# This removes both original and lagged VVIX columns
vvix_cols = [
    c for c in X.columns
    if "VVIX" in c  # Catches both "VVIX" and "VVIX_lag1"
]

X = X.drop(columns=vvix_cols)

Combine predictors and target. Recover X and y.

In [10]:
dataset = pd.concat(
    [X, y.rename("target")],
    axis=1,
)

dataset = dataset.dropna()

X = dataset.drop(columns="target")
y = dataset["target"]

Fit an Elastic Net/Ridge baseline using expanding-window cross-validation.

In [11]:
tscv = TimeSeriesSplit(
    n_splits=5,
)

pipeline = Pipeline(
    [
        ("scale", StandardScaler()),
        (
            "model",
            Ridge(
                alpha=100.0,
                # l1_ratio=0.5,
                max_iter=10000,
                random_state=42,
            ),
        ),
    ]
)

Evaluate:

In [12]:
oos_pred = pd.Series(index=y.index, dtype=float)

for train_idx, test_idx in tscv.split(X):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    pipeline.fit(X_train, y_train)

    oos_pred.iloc[test_idx] = pipeline.predict(X_test)

Compute metrics:

In [13]:
mask = oos_pred.notna()

rmse = np.sqrt(
    mean_squared_error(
        y[mask],
        oos_pred[mask],
    )
)

r2 = r2_score(
    y[mask],
    oos_pred[mask],
)

corr = np.corrcoef(
    y[mask],
    oos_pred[mask],
)[0, 1]

print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")
print(f"Corr : {corr:.6f}")

RMSE : 0.011921
R²   : -0.217343
Corr : 0.057278


In [14]:
coef = pd.Series(
    pipeline.named_steps["model"].coef_,
    index=X.columns,
)

coef = coef[coef != 0]

print(len(coef))
sorted = coef.sort_values(key=np.abs)
print(sorted.tail(20))
# sorted.to_csv(CACHE_PATH / "coef.csv")


368
Ret_5_lag1_^FTSE     -0.001275
Ret_10_lag1_^NYA     -0.001277
Ret_60_lag1_^VIX     -0.001308
Ret_5_^STOXX50E      -0.001328
Ret_60_^VIX           0.001380
Vol_10_^TWII         -0.001421
Ret_5_lag1_^GDAXI     0.001449
Vol_10_^RUT           0.001471
Vol_5_^STI            0.001495
Ret_10_^GSPTSE        0.001506
Vol_5_lag1_^TWII      0.001507
Ret_1_^VIX           -0.001572
Vol_5_^GDAXI          0.001582
Ret_1_^DJI           -0.001610
Ret_60_^TWII          0.001653
Vol_5_lag1_^GSPTSE   -0.001663
Ret_1_lag1_^NYA      -0.001767
Ret_1_^GSPC          -0.002066
Ret_20_lag1_^VIX     -0.002596
Ret_1_^GSPTSE        -0.003192
dtype: float64


In [15]:
print(y.describe())
print(oos_pred.describe())

count    4720.000000
mean        0.000415
std         0.012541
min        -0.119841
25%        -0.004189
50%         0.000716
75%         0.005971
max         0.115800
Name: target, dtype: float64
count    3930.000000
mean       -0.000042
std         0.005663
min        -0.028326
25%        -0.003325
50%         0.000009
75%         0.003507
max         0.031598
dtype: float64


In [16]:
print(np.max(np.abs(y)))
print(np.max(np.abs(oos_pred)))

0.1198405524039344
0.03159761254841671
